# 3. Capstone Pre-processing Training & Data Development

Random Forests are generally not sensitive to feature scales and are generally capable of handling high-dimensional data, Therfore, scaling and dimension reduction was not performed at this time. Also, no data was missing so no values were imputed.

In [1]:
# Libraries imported for this notebook.

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

In [2]:
# Read EDA_Data.xlsx into a dataframe, formatted dates, and indexed dates.

df = pd.read_excel('/Users/NJahns/Desktop/Bootcamp/Capstone_Two/Edited_Data/EDA_Data.xlsx', parse_dates=True, index_col=[0])

In [3]:
# Printed columns in data

for col in df.columns:
    print(col)

Sludge Volume Index
SE NH3
MLSS
RAS TSS
Alum Feed 1
Alum Feed 2
WAS Flow
Primary Sludge A Flow
Primary Sludge B Flow
F:M Ratio
RAS Flow
Avg ND Effluent Hourly Ammonia
Avg ND WAS Flow
Avg ND Aeration DO Cell 5 AVG
Avg ND Aeration DO Cell 7 AVG
Avg ND Aer. Avg Cell 7 Nitrate
Avg ND Average Basin Airflow
day
month
year


## Feature Engineering

### Domain-specific

In [4]:
# Changed 'WAS Flow' to Sludge flow to avaid confusion with 'Avg ND WAS Flow'

df.rename(columns={'WAS Flow': 'Activated Sludge Flow'}, inplace=True)

In [5]:
# Added a column that is 'Total Alum Feed'

df['Total Alum Feed'] = df['Alum Feed 1'] + df['Alum Feed 2']

In [6]:
# Added a column that is 'Total Sludge Flow'

df['Total Primary Sludge Flow'] = df['Primary Sludge A Flow'] + df['Primary Sludge B Flow']

In [7]:
# Added a column that is 'Average ND Aeration DO in Cells'

df['Average ND Aeration DO in Cells'] = df[['Avg ND Aeration DO Cell 5 AVG', 'Avg ND Aeration DO Cell 7 AVG']].mean(axis=1)

In [8]:
# Added a column that is 'MVLSS' - Mixed Liquor Volatile Suspended Solids. The volotile fraction is roughly 75%.

df['MVLSS'] = df['MLSS'] * 0.75

In [9]:
# Added a column that is 'MCRT' - Mean Cell Residence Time

df['MCRT'] = df['MVLSS'] / (df['Activated Sludge Flow'] + df['RAS Flow'])

In [10]:
# Added a column that is 'Sludge Age'

df['Sludge Age'] = df['MCRT'] / (1 + df['F:M Ratio'])

### Temporal

Temporal features such as day of the week, month, and year were created in previous steps.

#### Differencing

In [11]:
# Performed differencing on data

# Identified columns to exclude from differencing
exclude_columns = ['day', 'month', 'year']

# Performed first-order differencing on selected columns
first_differences = df.drop(columns=exclude_columns).diff().add_prefix('1st_')

# Performed second-order differencing on selected columns
second_differences = df.drop(columns=exclude_columns).diff().diff().add_prefix('2nd_')

# Concatenated the results with the original DataFrame
df = pd.concat([df, first_differences, second_differences], axis=1).dropna()

# Droped rows with NaN values
#df = result_df.dropna()

In [12]:
# Checked feature names

for column_title in df.columns:
    print(column_title)

Sludge Volume Index
SE NH3
MLSS
RAS TSS
Alum Feed 1
Alum Feed 2
Activated Sludge Flow
Primary Sludge A Flow
Primary Sludge B Flow
F:M Ratio
RAS Flow
Avg ND Effluent Hourly Ammonia
Avg ND WAS Flow
Avg ND Aeration DO Cell 5 AVG
Avg ND Aeration DO Cell 7 AVG
Avg ND Aer. Avg Cell 7 Nitrate
Avg ND Average Basin Airflow
day
month
year
Total Alum Feed
Total Primary Sludge Flow
Average ND Aeration DO in Cells
MVLSS
MCRT
Sludge Age
1st_Sludge Volume Index
1st_SE NH3
1st_MLSS
1st_RAS TSS
1st_Alum Feed 1
1st_Alum Feed 2
1st_Activated Sludge Flow
1st_Primary Sludge A Flow
1st_Primary Sludge B Flow
1st_F:M Ratio
1st_RAS Flow
1st_Avg ND Effluent Hourly Ammonia
1st_Avg ND WAS Flow
1st_Avg ND Aeration DO Cell 5 AVG
1st_Avg ND Aeration DO Cell 7 AVG
1st_Avg ND Aer. Avg Cell 7 Nitrate
1st_Avg ND Average Basin Airflow
1st_Total Alum Feed
1st_Total Primary Sludge Flow
1st_Average ND Aeration DO in Cells
1st_MVLSS
1st_MCRT
1st_Sludge Age
2nd_Sludge Volume Index
2nd_SE NH3
2nd_MLSS
2nd_RAS TSS
2nd_Alum Feed

#### Lag features

I created lagged versions of target metric and features of one through seven days since SVI tends to change over the course of weeks. Trying different lag values or determining lag values based on data analysis could be performed for future versions of the model in order to determine the time period that is most predictive. Creating rolling statistics could also be explored.

In [13]:
# Lagged features by 1 through 7 days.

explanatory_vars = df.columns[~df.columns.isin(['day', 'month', 'year'])]
lagged_columns = []

lagged_df = pd.DataFrame()

for lag in range(1, 8):
    lagged_columns.extend([f'{var}_lag_{lag}' for var in explanatory_vars])
    lagged_data = {f"{var}_lag_{lag}": df[var].shift(lag) for var in explanatory_vars}
    lagged_df = pd.concat([lagged_df, pd.DataFrame(lagged_data)], axis=1)

df = pd.concat([df, lagged_df], axis=1)

In [14]:
# Dropped rows containng NaN as a result of lagged features.

df.dropna(inplace=True)

In [15]:
# Printed list of column titles to check results of lagging.

for column_title in df.columns:
    print(column_title)

Sludge Volume Index
SE NH3
MLSS
RAS TSS
Alum Feed 1
Alum Feed 2
Activated Sludge Flow
Primary Sludge A Flow
Primary Sludge B Flow
F:M Ratio
RAS Flow
Avg ND Effluent Hourly Ammonia
Avg ND WAS Flow
Avg ND Aeration DO Cell 5 AVG
Avg ND Aeration DO Cell 7 AVG
Avg ND Aer. Avg Cell 7 Nitrate
Avg ND Average Basin Airflow
day
month
year
Total Alum Feed
Total Primary Sludge Flow
Average ND Aeration DO in Cells
MVLSS
MCRT
Sludge Age
1st_Sludge Volume Index
1st_SE NH3
1st_MLSS
1st_RAS TSS
1st_Alum Feed 1
1st_Alum Feed 2
1st_Activated Sludge Flow
1st_Primary Sludge A Flow
1st_Primary Sludge B Flow
1st_F:M Ratio
1st_RAS Flow
1st_Avg ND Effluent Hourly Ammonia
1st_Avg ND WAS Flow
1st_Avg ND Aeration DO Cell 5 AVG
1st_Avg ND Aeration DO Cell 7 AVG
1st_Avg ND Aer. Avg Cell 7 Nitrate
1st_Avg ND Average Basin Airflow
1st_Total Alum Feed
1st_Total Primary Sludge Flow
1st_Average ND Aeration DO in Cells
1st_MVLSS
1st_MCRT
1st_Sludge Age
2nd_Sludge Volume Index
2nd_SE NH3
2nd_MLSS
2nd_RAS TSS
2nd_Alum Feed

In [16]:
# Saved to Excel
df.to_excel('/Users/NJahns/Desktop/Bootcamp/Capstone_Two/Edited_Data/Pre_Process_Train.xlsx', index=True)

I did not include splitting the data into testing and training datasets in this workbook becasue it makes more sense to include those steps with model development as they are mixed into the model development steps.